In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import duckdb


In [4]:

# Load the Parquet file using DuckDB
con = duckdb.connect()
df = con.execute("""
    SELECT *
    FROM read_parquet('nyiso_dataset/**/*.parquet')
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
#need to double check zone coords
zone_coords = {
    "CAPITL": (42.65, -73.75),
    "CENTRL": (43.05, -76.15),
    "DUNWOD": (41.01, -73.78),
    "GENESE": (43.17, -77.61),
    "HUD VL": (41.70, -73.93),
    "MHK VL": (42.10, -75.91),
    "MILLWD": (41.13, -73.78),
    "N.Y.C.": (40.71, -74.01),
    "NORTH": (44.70, -73.45),
    "WEST": (42.89, -78.87)
}

In [6]:
df

,Time Stamp,Time Zone,Name,PTID,Load,month,year
0,2001-05-28 00:00:00,EDT,CAPITL,61757.0,894.0000,05,2001
1,2001-05-28 00:00:00,EDT,CENTRL,61754.0,1346.0000,05,2001
2,2001-05-28 00:00:00,EDT,DUNWOD,61760.0,415.0000,05,2001
3,2001-05-28 00:00:00,EDT,GENESE,61753.0,746.0000,05,2001
4,2001-05-28 00:00:00,EDT,HUD VL,61758.0,978.0000,05,2001
...,...,...,...,...,...,...,...
29327978,2025-10-03 23:55:00,EDT,MHK VL,61756.0,671.4786,10,2025
29327979,2025-10-03 23:55:00,EDT,MILLWD,61759.0,206.7461,10,2025
29327980,2025-10-03 23:55:00,EDT,N.Y.C.,61761.0,4669.4595,10,2025
29327981,2025-10-03 23:55:00,EDT,NORTH,61755.0,567.2412,10,2025


In [7]:
df_total_load = df.groupby("Time Stamp", as_index=False)["Load"].sum()
df_total_load.rename(columns={"Time Stamp": "Time"}, inplace=True)


In [8]:
df_total_load

,Time,Load
0,2001-05-26 00:00:00,13859.0000
1,2001-05-26 00:00:50,13761.0000
2,2001-05-26 00:05:20,13718.0000
3,2001-05-26 00:06:50,13622.0000
4,2001-05-26 00:11:50,13551.0000
...,...,...
2713135,2025-10-08 12:55:00,15401.3201
2713136,2025-10-08 13:00:00,15400.6558
2713137,2025-10-08 13:05:00,15490.6380
2713138,2025-10-08 13:10:00,15456.0476


In [9]:
# Ensure the 'Time' column is a datetime type and set it as the index
df_total_load['Time'] = pd.to_datetime(df_total_load['Time'])
df_total_load.set_index('Time', inplace=True)

# Average hourly load
df_hourly = df_total_load.resample('1H').mean().reset_index()
df_hourly = df_hourly.dropna().reset_index()



C:\Users\sarah\AppData\Local\Temp\ipykernel_71732\3431484118.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df_total_load.resample('1H').mean().reset_index()


In [10]:
df_hourly

,index,Time,Load
0,0,2001-05-26 00:00:00,13388.875000
1,1,2001-05-26 01:00:00,12663.000000
2,2,2001-05-26 02:00:00,12152.000000
3,3,2001-05-26 03:00:00,11888.866667
4,4,2001-05-26 04:00:00,11824.928571
...,...,...,...
212822,213633,2025-10-08 09:00:00,16797.797893
212823,213634,2025-10-08 10:00:00,16232.384442
212824,213635,2025-10-08 11:00:00,15846.538767
212825,213636,2025-10-08 12:00:00,15499.850792


In [11]:
df_total_load = df_hourly

In [12]:
# Split the data based on the year
train_data = df_total_load[df_total_load['Time'].dt.year.between(2001, 2021)]['Load']
val_data = df_total_load[df_total_load['Time'].dt.year == 2022]['Load']
test_data = df_total_load[df_total_load['Time'].dt.year.isin([2023, 2024, 2025])]['Load']

# Print the sizes of each split
print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Testing data size: {len(test_data)}")

Training data size: 179793
Validation data size: 8759
Testing data size: 24275


In [13]:
scaler = StandardScaler()
train_scaled = scaler.fit_transform(pd.DataFrame(train_data))
val_scaled = scaler.transform(pd.DataFrame(val_data))
test_scaled = scaler.transform(pd.DataFrame(test_data))

In [14]:
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        v = X[i:i + time_steps]
        Xs.append(v)
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 5
X_train, y_train = create_dataset(train_scaled, train_scaled, TIME_STEPS)
X_val, y_val = create_dataset(val_scaled, val_scaled, TIME_STEPS)
X_test, y_test = create_dataset(test_scaled, test_scaled, TIME_STEPS)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(179788, 5, 1) (179788, 1)
(8754, 5, 1) (8754, 1)
(24270, 5, 1) (24270, 1)


In [15]:
print("NaNs in X_train:", np.isnan(X_train).sum())
print("NaNs in X_val:", np.isnan(X_val).sum())
print("NaNs in y_train:", np.isnan(y_train).sum())
print("NaNs in y_val:", np.isnan(y_val).sum())

NaNs in X_train: 0
NaNs in X_val: 0
NaNs in y_train: 0
NaNs in y_val: 0


In [16]:
subset_size = 40000
val_subset = 8749
X_train_sub = X_train[:subset_size]
y_train_sub = y_train[:subset_size]
X_val_sub = X_val[:val_subset]
y_val_sub = y_val[:val_subset]


In [17]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
# best_mae = float('inf')
# best_model = None

# for C in [0.1, 1, 10]:
#     for gamma in ['scale', 0.01, 0.001]:
#         for epsilon in [0.01, 0.1, 0.5, 1.0]:
#             print(f"C={C}, gamma={gamma}, epsilon={epsilon}")
#             model = SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon)
#             model.fit(X_train_sub.reshape(X_train_sub.shape[0], -1), y_train_sub)
#             preds = model.predict(X_val_sub.reshape(X_val_sub.shape[0], -1))
#             mae = mean_absolute_error(y_val_sub, preds)
#             print(f"val MAE={mae:.3f}")

#             if mae < best_mae:
#                 best_mae = mae
#                 best_model = model

# print("Best params found:", best_model.get_params())


Best params found: {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}

Best mae: 0.04994650252799576

In [18]:
best_params = {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}
best_model = SVR(**best_params)
best_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)

C:\Users\sarah\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


SVR(C=10, epsilon=0.01, gamma=0.01)

In [19]:

# Make predictions
train_pred = best_model.predict(X_train.reshape(X_train.shape[0], -1))
val_pred = best_model.predict(X_val.reshape(X_val.shape[0], -1))
test_pred = best_model.predict(X_test.reshape(X_test.shape[0], -1))

# Inverse scaling
train_pred_inv = scaler.inverse_transform(train_pred.reshape(-1, 1))
y_train_inv = scaler.inverse_transform(y_train.reshape(-1, 1))

val_pred_inv = scaler.inverse_transform(val_pred.reshape(-1, 1))
y_val_inv = scaler.inverse_transform(y_val.reshape(-1, 1))

test_pred_inv = scaler.inverse_transform(test_pred.reshape(-1, 1))
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))


In [20]:
# Evaluate the model
mae_train = mean_absolute_error(y_train_inv, train_pred_inv)
mae_test = mean_absolute_error(y_test_inv, test_pred_inv)
print("Mean Absolute Error on Training Data:", mae_train)
print("Mean Absolute Error on Testing Data:", mae_test)

Mean Absolute Error on Training Data: 212.21347312345026
Mean Absolute Error on Testing Data: 170.34232309986712


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np

# Take a subset for speed (first 1000 points)
y_true = y_test_inv
y_pred = test_pred_inv
x = np.arange(len(y_true))

fig, ax = plt.subplots(figsize=(10,6))
line_true, = ax.plot([], [], label="True Values", color="blue", alpha=0.7)
line_pred, = ax.plot([], [], label="Predictions", color="red", alpha=0.7)
ax.set_xlim(0, len(y_true))
ax.set_ylim(min(y_true.min(), y_pred.min())*0.95, max(y_true.max(), y_pred.max())*1.05)
ax.set_xlabel("Index")
ax.set_ylabel("Load")
ax.set_title("True Values vs Predictions Test")
ax.legend()

def update(frame):
    line_true.set_data(x[:frame], y_true[:frame])
    line_pred.set_data(x[:frame], y_pred[:frame])
    return line_true, line_pred

ani = FuncAnimation(fig, update, frames=len(x), interval=20, blit=True)
ani.save("moving_plot.gif", writer='pillow', fps=30)



In [22]:
# from datetime import datetime
# import matplotlib.pyplot as plt
# import meteostat
# from meteostat import Point, Daily, Hourly

# start = datetime(2018, 1, 1, 0, 0)
# end = datetime(2018, 1, 1, 12, 0)

# # Create Point for Vancouver, BC
# vancouver = Point(42.65, -73.75)

# # Get daily data for 2018
# data = Hourly(vancouver, start, end)
# data = data.fetch()

# # Plot line chart including average, minimum and maximum temperature
# data